In [6]:
# 单元格 1: 安装依赖
!pip install -U FlagEmbedding transformers datasets accelerate -q

import torch
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
print(f"当前 GPU 设备: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}")


CUDA 是否可用: True
当前 GPU 设备: NVIDIA A10


In [2]:
# 单元格 2: 生成 50 条包含 -1 到 3 分细粒度标签的回归训练数据
import json
import random

def generate_regression_web_data(output_path="train_regression_data.jsonl", num_queries=50):
    topics = [
        "如何申请新加坡工作签证", "2026年最新AI大模型排名", 
        "OCI云服务器A10实例配置教程", "Python多线程深度学习加速",
        "企业级大模型RAG精排优化方案", "混合检索Hybrid Search得分计算"
    ]
    domains = ["gov.sg", "techcrunch.com", "oracle.com", "github.com", "seo-spam-site.biz", "v2ex.com"]
    
    with open(output_path, "w", encoding="utf-8") as f:
        for i in range(num_queries):
            topic = random.choice(topics)
            query = f"{topic} 精准搜索第{i+1}次"
            
            # 定义 5 个不同相关度阶梯的网页内容
            scenarios = [
                {"label": 3.0, "title": "【官方权威指南】", "domain": "official-gov-or-vendor.org", "body": "这是满足用户核心意图的官方最权威内容，包含2026年最新政策、官方详尽步骤以及无瑕疵的解答。"},
                {"label": 2.0, "title": "优质技术博客深度解析：", "domain": "medium-tech-blog.com", "body": "这是一篇来自第三方社区或资深工程师的高分实操总结，给出了完整的代码或方案细节，极具参考价值。"},
                {"label": 1.0, "title": "泛谈相关概念引入 - ", "domain": "wikipedia-style.org", "body": "本文仅泛泛地科普了相关的底层的基本概念，字面上存在部分重合，但无法直接解决用户当前的核心痛点。"},
                {"label": 0.0, "title": "今日无关数码娱乐科技快讯", "domain": "daily-news.com", "body": "今天天气晴朗，科技界发生了一件无关的大事。这是一篇完全不沾边的通用新闻，用来作为基线负样本。"},
                {"label": -1.0, "title": "【点击直达领取福利】", "domain": "seo-spam-site.biz", "body": "购买 便宜 优惠 强力推荐 各种关键词恶意堆砌，时效性严重过期，点进去全都是垃圾SEO广告弹窗。"}
            ]
            
            # 为当前 Query 写入 5 个不同得分的 Doc，总共 50 个 Query 将产生 250 条训练对
            for scene in scenarios:
                text_content = f"标题：{scene['title']}{topic} | 域名：{scene['domain']} | 正文：{scene['body']}"
                data_line = {
                    "query": query,
                    "text": text_content,
                    "label": scene["label"]
                }
                f.write(json.dumps(data_line, ensure_ascii=False) + "\n")
                
    print(f"成功扩展生成 {num_queries} 个意图组（共 {num_queries * 5} 条单步回归对），已保存至 {output_path}")

# 执行数据生成
generate_regression_web_data("train_regression_data.jsonl", num_queries=50)


成功扩展生成 50 个意图组（共 250 条单步回归对），已保存至 train_regression_data.jsonl


In [3]:
# 单元格 3: 基于 BAAI/bge-reranker-v2-m3 启动回归微调
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from datasets import load_dataset

# 1. 加载回归模型与分词器（显式指定 num_labels=1）
model_name = "BAAI/bge-reranker-v2-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1, 
    ignore_mismatched_sizes=True
)

# 2. 加载与分词处理
dataset = load_dataset("json", data_files="train_regression_data.jsonl")

def tokenize_function(examples):
    # Cross-Encoder 标准拼接：[CLS] Query [SEP] Text [SEP]
    return tokenizer(
        examples["query"], 
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=512
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. 针对单卡 A10 (24GB) 优化训练超参数
training_args = TrainingArguments(
    output_dir="./web_reranker_regression_out",
    learning_rate=2e-5,                  # 排序精排黄金学习率
    num_train_epochs=3,                  # 数据量小时可以适当多跑 2-3 个 epoch 促进收敛
    per_device_train_batch_size=4,       # A10 跑 512 长度，BatchSize=4 显存非常安全
    gradient_accumulation_steps=2,       # 梯度累加，等效总 BatchSize = 8
    fp16=True,                           # 充分利用 A10 的 FP16 混合精度算力加速
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",                  # 50条小样本直接跑完再保存
    remove_unused_columns=True,          # 自动剔除 query, text 等非 tensor 列
    report_to="none"
)

# 4. 启动训练
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
)

print("正在 A10 GPU 上以【-1至3分回归打分模式】训练模型...")
trainer.train()

# 5. 保存微调后的精排模型
model.save_pretrained("./web_reranker_regression_final")
tokenizer.save_pretrained("./web_reranker_regression_final")
print("训练结束！模型已成功保存至 ./web_reranker_regression_final")


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (6 outputs left out)

正在 A10 GPU 上以【-1至3分回归打分模式】训练模型...


Step,Training Loss
10,9.558023
20,3.321555
30,0.575819
40,0.244491
50,0.243173
60,0.103018
70,0.074073
80,0.071953
90,0.044006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

训练结束！模型已成功保存至 ./web_reranker_regression_final


In [5]:
# 单元格 4: 升级版 - 自动计算 NDCG@5 和 MRR@10 排序指标
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ==========================================
# 1. 核心数学指标计算函数 (支持 -1~3分 细粒度标签)
# ==========================================
def calculate_query_metrics(pred_scores, true_labels, ndcg_k=5, mrr_k=10):
    """
    计算单个 Query 的 NDCG 和 MRR
    """
    # 按照模型预测的得分从高到低进行降序排列，获取排序后的索引
    rank_indices = np.argsort(pred_scores)[::-1]
    
    # 根据模型的排序结果，对真实的标签进行重排
    reranked_true_labels = np.array(true_labels)[rank_indices]
    
    # ------------------------------------------
    # A. 计算 NDCG@K (标量回归标准公式)
    # ------------------------------------------
    # 为了防止 -1 分在指数计算中带来数学异常，将所有标签平移到正数区间进行 DCG 计算
    # 行业标准公式: Gain = 2^label - 1 (由于最大标签是3，这里平移后计算极为稳定)
    def compute_dcg(labels_list, k):
        dcg = 0.0
        for rank, label in enumerate(labels_list[:k]):
            # 将 -1~3 分平移到 0~4 分计算增益
            gain = (2 ** (label + 1)) - 1
            dcg += gain / np.log2(rank + 2)
        return dcg

    # 计算模型实际排序的 DCG
    actual_dcg = compute_dcg(reranked_true_labels, ndcg_k)
    
    # 计算理想状态下（真实标签直接降序排列）的 IDCG
    ideal_labels = sorted(true_labels, reverse=True)
    ideal_dcg = compute_dcg(ideal_labels, ndcg_k)
    
    # 归一化得到 NDCG
    ndcg = (actual_dcg / ideal_dcg) if ideal_dcg > 0 else 0.0

    # ------------------------------------------
    # B. 计算 MRR@K (网页搜索标准：找到第一个完美/高度相关结果的倒数排名)
    # ------------------------------------------
    mrr = 0.0
    for rank, label in enumerate(reranked_true_labels[:mrr_k]):
        # 在网页精排场景中，只有 2分(优质) 或 3分(完美) 才被视为真正命中了用户意图
        if label >= 2.0:
            mrr = 1.0 / (rank + 1)
            break  # 找到第一个满足条件的答案后立即截断
            
    return ndcg, mrr

# ==========================================
# 2. 构造包含多个意图组的离线多维测试集
# ==========================================
evaluation_dataset = [
    {
        "query": "OCI云服务器A10实例配置教程",
        "docs": [
            "标题：什么是云计算服务器及GPU的基础概念科普 | 域名：wiki.org",
            "标题：【点我限时白嫖A10】甲骨文优惠券永久免费翻墙 | 域名：seo-spam.biz",
            "标题：玩转甲骨文云：OCI A10 显卡避坑调优随笔 | 域名：v2ex.com",
            "标题：【Oracle官方】OCI Compute A10 GPU 实例全套配置与驱动安装指南 | 域名：oracle.com",
            "标题：今日推荐：如何在家自制一杯冰美式咖啡 | 域名：coffee.com"
        ],
        "labels": [1.0, -1.0, 2.0, 3.0, 0.0] # 真实的 -1~3 分标准标签
    },
    {
        "query": "2026年最新AI大模型排名",
        "docs": [
            "标题：【2026权威发布】全球AI大模型技术路线与综合排名 | 域名：techcrunch.com",
            "标题：点击查看相关论坛讨论-大模型哪个好 | 域名：v2ex.com",
            "标题：人工智能历史发展简史回顾 | 域名：history.org",
            "标题：SEO低质垃圾站-最新大模型批发买卖 | 域名：bad-seo.net"
        ],
        "labels": [3.0, 2.0, 1.0, -1.0]
    }
]

# ==========================================
# 3. 载入模型并启动自动化指标评测
# ==========================================
eval_model_path = "./web_reranker_regression_final"
eval_tokenizer = AutoTokenizer.from_pretrained(eval_model_path)
eval_model = AutoModelForSequenceClassification.from_pretrained(eval_model_path).cuda().eval()

all_ndcg_scores = []
all_mrr_scores = []

print("🚀 开始在 OCI A10 上进行 Rerank 模型自动化指标评测...\n" + "="*60)

with torch.no_grad():
    for item in evaluation_dataset:
        query = item["query"]
        docs = item["docs"]
        labels = item["labels"]
        
        # 构造 Cross-Encoder 输入对
        pairs = [[query, doc] for doc in docs]
        inputs = eval_tokenizer(pairs, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        
        # 模型前向传播计算标量得分
        outputs = eval_model(**inputs)
        pred_scores = outputs.logits.view(-1).cpu().numpy()
        
        # 计算该 Query 的排序指标
        ndcg_5, mrr_10 = calculate_query_metrics(pred_scores, labels, ndcg_k=5, mrr_k=10)
        all_ndcg_scores.append(ndcg_5)
        all_mrr_scores.append(mrr_10)
        
        # 实时打印每个案例的微观排序表现
        print(f"搜索 Query: '{query}'")
        print(f" -> 当前意图组指标: NDCG@5 = {ndcg_5:.4f} | MRR@10 = {mrr_10:.4f}")
        # 展示模型重排后的前 2 名结果
        top_indices = np.argsort(pred_scores)[::-1]
        print(f" -> 模型排在第 1 名结果: [预估 {pred_scores[top_indices[0]]:.2f}分] {docs[top_indices[0]][:40]}...")
        print(f" -> 模型排在第 2 名结果: [预估 {pred_scores[top_indices[1]]:.2f}分] {docs[top_indices[1]][:40]}...")
        print("-" * 60)

# ==========================================
# 4. 输出整个评测集的全局宏观盘点指标
# ==========================================
final_ndcg = np.mean(all_ndcg_scores)
final_mrr = np.mean(all_mrr_scores)

print("\n" + "==== 📊 全局宏观模型排序能力盘点 ====")
print(f"评估集平均 NDCG@5:  {final_ndcg:.4f} (通用网页搜索工业级合格线 > 0.80)")
print(f"评估集平均 MRR@10:  {final_mrr:.4f} (通用网页搜索工业级合格线 > 0.75)")
print("=========================================")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

🚀 开始在 OCI A10 上进行 Rerank 模型自动化指标评测...
搜索 Query: 'OCI云服务器A10实例配置教程'
 -> 当前意图组指标: NDCG@5 = 0.9979 | MRR@10 = 1.0000
 -> 模型排在第 1 名结果: [预估 1.77分] 标题：【Oracle官方】OCI Compute A10 GPU 实例全套配置与...
 -> 模型排在第 2 名结果: [预估 0.36分] 标题：玩转甲骨文云：OCI A10 显卡避坑调优随笔 | 域名：v2ex.com...
------------------------------------------------------------
搜索 Query: '2026年最新AI大模型排名'
 -> 当前意图组指标: NDCG@5 = 0.9901 | MRR@10 = 1.0000
 -> 模型排在第 1 名结果: [预估 2.03分] 标题：【2026权威发布】全球AI大模型技术路线与综合排名 | 域名：techc...
 -> 模型排在第 2 名结果: [预估 0.00分] 标题：点击查看相关论坛讨论-大模型哪个好 | 域名：v2ex.com...
------------------------------------------------------------

==== 📊 全局宏观模型排序能力盘点 ====
评估集平均 NDCG@5:  0.9940 (通用网页搜索工业级合格线 > 0.80)
评估集平均 MRR@10:  1.0000 (通用网页搜索工业级合格线 > 0.75)
